# Batch FASTQ reader

Scan the selected FASTQ folder (no subfolders), generate a length-sorted CSV and histogram for every file, and align to a matching FASTA or SnapGene `.dna` reference when available.

Match references by the complete leading number: `1132.fastq.gz` matches `1132 LPPC EE R-S.dna`. Multiple matching references produce QC only and an ambiguity in the batch report. Missing references also produce QC only.

All outputs go directly into one separate output folder. Completed samples are skipped; set `OVERWRITE = True` to rebuild them. Failed or changed samples with existing outputs also require overwrite. The original input files are never modified.

Install Python dependencies with `python -m pip install -r requirements-notebook.txt`, and install `minimap2` and `samtools` separately for alignment. Open this notebook with its working directory set to the repository folder. In the prepared cloud environment, first run `source /workspace/fastq-reader-local/activate.sh`.


In [ ]:
from pathlib import Path

FASTQ_DIR = Path("/path/to/fastq_folder")
REFERENCE_DIR = None  # None searches FASTQ_DIR; otherwise Path("/path/to/reference_folder")
OUTPUT_DIR = Path("/path/to/output_folder")  # all results in this one folder

OVERWRITE = False
QC_ONLY = False  # True skips reference search and alignment
BIN_SIZE_BP = 2000
LOG_Y = False
THREADS = 4


In [ ]:
from batch_fastq_reader import run_batch

report = run_batch(
    fastq_dir=FASTQ_DIR,
    reference_dir=REFERENCE_DIR,
    output_dir=OUTPUT_DIR,
    overwrite=OVERWRITE,
    qc_only=QC_ONLY,
    bin_size=BIN_SIZE_BP,
    log_y=LOG_Y,
    threads=THREADS,
)
display(report)


Review the `status` and `message` columns for missing references, ambiguous matches, or failures. QC outputs are kept if alignment fails; the other samples continue processing. `batch_report.csv` contains the latest run's report.

Each FASTQ is processed separately, even if several filenames have the same sample number. Outputs normally use the FASTQ stem (`1132_summary.csv`, `1132_length_hist.png`, `1132_sorted.bam`). When multiple FASTQs share an identical stem, their full filenames become their output prefixes to prevent collisions.
